# Event-Driven Multi-Agent Architecture | Multi-Agent Collaboration

In [1]:
from langchain_openai import ChatOpenAI
from typing import Dict, List, Callable
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from datetime import datetime
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
@dataclass
class Event:
    event_type: str
    payload: Dict
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())

class EventBus:
    """Simple in-memory event bus for agent coordination."""
    def __init__(self):
        self._subscribers: Dict[str, List[Callable]] = {}
        self._event_log: List[Dict] = []

    def subscribe(self, event_type: str, handler: Callable):
        self._subscribers.setdefault(event_type, []).append(handler)

    def publish(self, event: Event) -> List[str]:
        """Publish event and collect results from all subscribers."""
        handlers = self._subscribers.get(event.event_type, [])
        self._event_log.append({
            "event": event.event_type,
            "payload": event.payload,
            "subscribers": len(handlers),
            "timestamp": event.timestamp,
        })
        # Execute all handlers in parallel
        results = []
        if handlers:
            with ThreadPoolExecutor() as executor:
                futures = [executor.submit(h, event) for h in handlers]
                results = [f.result() for f in futures]
        return results

In [5]:
# Create event bus
bus = EventBus()

# Define event-driven agents
def welcome_agent(event: Event) -> str:
    response = model.invoke(
        f"You are a welcome email agent. Draft a personalized welcome email for:\n"
        f"Customer: {event.payload.get('name', 'User')}\n"
        f"Plan: {event.payload.get('plan', 'free')}\n"
        f"Keep it brief and warm."
    )
    return f"[Welcome Agent] {response.content}"

def analytics_agent(event: Event) -> str:
    payload = event.payload
    return (
        f"[Analytics Agent] Tracked: new_signup | "
        f"name={payload.get('name')} | plan={payload.get('plan')} | "
        f"source={payload.get('source', 'unknown')} | "
        f"timestamp={event.timestamp}"
    )

def onboarding_agent(event: Event) -> str:
    response = model.invoke(
        f"You are an onboarding agent. Create a 3-step onboarding checklist for:\n"
        f"Customer: {event.payload.get('name', 'User')}\n"
        f"Plan: {event.payload.get('plan', 'free')}\n"
        f"Be specific and actionable."
    )
    return f"[Onboarding Agent] {response.content}"

In [6]:
# Subscribe agents to events
bus.subscribe("customer.signup", welcome_agent)
bus.subscribe("customer.signup", analytics_agent)
bus.subscribe("customer.signup", onboarding_agent)

# Publish an event
event = Event(
    event_type="customer.signup",
    payload={"name": "Alice Johnson", "plan": "pro", "source": "website"},
)
results = bus.publish(event)

for result in results:
    print(f"\n{'='*50}\n{result}")

print(f"\nEvent log (signup): {bus._event_log}")


[Welcome Agent] Subject: Welcome to the Pro Plan Family, Alice!

Hi Alice,

Warm greetings and a big welcome to our community! We’re thrilled to have you on board with the Pro Plan.

Your journey with us is just beginning, and we’re here to make sure you get the most out of your experience. If you have any questions, need assistance, or simply want to explore more features, don’t hesitate to reach out.

Once again, welcome! Here’s to achieving great things together.

Best regards,

[Your Name]  
[Your Company's Name]  
[Contact Information]

[Analytics Agent] Tracked: new_signup | name=Alice Johnson | plan=pro | source=website | timestamp=2026-04-09T11:02:08.970008

[Onboarding Agent] Certainly! Here's a detailed 3-step onboarding checklist for Alice Johnson, who has subscribed to the Pro plan:

### Step 1: Account Setup and Verification
- **1.1: Confirm Account Details**
  - Ensure that Alice's account is set up with the correct information: name, email (alice.johnson@example.com), a

In [7]:
# === Dynamic subscriber addition ===
# Key benefit: adding new subscribers requires ZERO changes to existing agents or the publisher
def loyalty_agent(event: Event) -> str:
    """Added AFTER the first event -- demonstrates dynamic subscriber addition."""
    return (
        f"[Loyalty Agent] Enrolled {event.payload.get('name')} in "
        f"{event.payload.get('plan', 'free')}-tier rewards program"
    )

bus.subscribe("customer.signup", loyalty_agent)  # New subscriber, no existing code changed

# === Second event type: shows the event bus handles multiple event types ===
def fulfillment_agent(event: Event) -> str:
    return f"[Fulfillment Agent] Processing order #{event.payload.get('order_id')} for {event.payload.get('name')}"

def notification_agent(event: Event) -> str:
    return f"[Notification Agent] Sent confirmation email for order #{event.payload.get('order_id')}"

bus.subscribe("purchase.completed", fulfillment_agent)
bus.subscribe("purchase.completed", notification_agent)

# Publish a different event type
purchase_event = Event(
    event_type="purchase.completed",
    payload={"name": "Alice Johnson", "order_id": "ORD-4821", "total": 299.99},
)
purchase_results = bus.publish(purchase_event)

print("\n" + "="*50 + "\n--- Purchase Event Results ---")
for r in purchase_results:
    print(r)

# A second signup now triggers 4 agents (including the dynamically added loyalty_agent)
event2 = Event(
    event_type="customer.signup",
    payload={"name": "Bob Smith", "plan": "enterprise", "source": "referral"},
)
results2 = bus.publish(event2)
print("\n--- Second signup (4 subscribers now) ---")
for r in results2:
    print(f"\n{r}")

print(f"\nFull event log: {bus._event_log}")


--- Purchase Event Results ---
[Fulfillment Agent] Processing order #ORD-4821 for Alice Johnson
[Notification Agent] Sent confirmation email for order #ORD-4821

--- Second signup (4 subscribers now) ---

[Welcome Agent] Subject: Welcome to Our Enterprise Family, Bob!

Hi Bob,

Welcome aboard! We're thrilled to have you join our enterprise plan family.

Your journey with us is just beginning, and we're eager to support your goals every step of the way. Our team is here to ensure you get the most out of your plan.

If you have any questions or need assistance, feel free to reach out. We're here for you!

Warm regards,

[Your Name]  
[Your Company]  
[Contact Information]

[Analytics Agent] Tracked: new_signup | name=Bob Smith | plan=enterprise | source=referral | timestamp=2026-04-09T11:02:08.979209

[Onboarding Agent] Certainly! Here's a tailored 3-step onboarding checklist for Bob Smith, who is subscribing to the Enterprise plan:

---

### Onboarding Checklist for Bob Smith - Enterpr